In [1]:
#mount gdrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


<a id="imported-libraries"></a>
## <font style = 'color:black'>001 - Imported libraries</font>


In [2]:
# Import all relevant libraries
import pandas as pd # Tables
import numpy as np # math calculation
import matplotlib.pyplot as plt # Visualisation
import seaborn as sns # Visualisation

# Additional libraries
from pathlib import Path # For getting filename in OS
from scipy.stats import skew # For Linear Regression to be symmetric

# Feature Importance & Predictive Model Libraries
from matplotlib.ticker import PercentFormatter # Convert format axis tick labels as percentages
from sklearn.model_selection import train_test_split # training 80%, testing 20%
from sklearn.linear_model import LinearRegression, Ridge # Model 2, Model 3
from sklearn.ensemble import RandomForestRegressor # Model 4
from sklearn.dummy import DummyRegressor # Model 1
from sklearn.preprocessing import LabelEncoder # Encode categorical to numeric
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neighbors import BallTree

# Libraries for API calls
import requests
import json
from tqdm import tqdm #create a progress bar for an iterable, typically a loop
import time #provides various time-related functions

# This makes the picture stay on the page! Not open a new one
%matplotlib inline

## <font style = 'color:black'>002 - Import CSV and Merge</font>
---
Check CSV files for column info and merge them.


1. Import .csv files and check info
2. Merge and check head and tail


In [3]:
# Define API key as a string here for later use:
my_key = 'x'
headers = {"Authorization": my_key}

In [4]:
#import csv
prices2015 = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/Main CSV/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv')
prices2017 = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/Main CSV/Resale flat prices based on registration date from Jan-2017 onwards.csv')


In [5]:
prices = pd.concat([prices2015, prices2017], sort=False)
prices.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,07 TO 09,60.0,Improved,1986,70,255000.0
1,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,01 TO 03,68.0,New Generation,1981,65,275000.0
2,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,01 TO 03,69.0,New Generation,1980,64,285000.0
3,2015-01,ANG MO KIO,3 ROOM,446,ANG MO KIO AVE 10,01 TO 03,68.0,New Generation,1979,63,290000.0
4,2015-01,ANG MO KIO,3 ROOM,557,ANG MO KIO AVE 10,07 TO 09,68.0,New Generation,1980,64,290000.0


In [6]:
prices.tail()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
223410,2026-01,YISHUN,5 ROOM,626,YISHUN ST 61,07 TO 09,121.0,Improved,1987,60 years 05 months,745888.0
223411,2026-01,YISHUN,EXECUTIVE,405,YISHUN AVE 6,04 TO 06,142.0,Apartment,1988,61 years 08 months,880000.0
223412,2026-01,YISHUN,EXECUTIVE,325,YISHUN CTRL,10 TO 12,146.0,Maisonette,1988,61 years 11 months,888000.0
223413,2026-01,YISHUN,EXECUTIVE,360,YISHUN RING RD,07 TO 09,142.0,Apartment,1988,61 years 07 months,865888.0
223414,2026-01,YISHUN,EXECUTIVE,643,YISHUN ST 61,10 TO 12,142.0,Apartment,1987,60 years 09 months,825000.0


In [7]:
# Check the columns and rows
print(f'combined price shape: {prices.shape}')
print(f'combined price data types:\n{prices.dtypes}\n')

combined price shape: (260568, 11)
combined price data types:
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
remaining_lease         object
resale_price           float64
dtype: object



In [8]:
#combine block and streetname to get address
prices['address'] = prices['block'] + ' ' + prices['street_name']
all_address = list(prices['address'])
unique_address = list(set(all_address))

print('Unique addresses:', len(unique_address))

Unique addresses: 9697


In [9]:
unique_address[:10]

['566 HOUGANG ST 51',
 '241 KIM KEAT LINK',
 '858 TAMPINES AVE 5',
 '38C BENDEMEER RD',
 '504D YISHUN ST 51',
 '345 CLEMENTI AVE 5',
 '411 CHOA CHU KANG AVE 3',
 '337 HOUGANG AVE 7',
 '936 JURONG WEST ST 91',
 '221C BEDOK CTRL']

In [10]:
#print 2015 to 2025 prices to csv
# prices.to_csv(
#     "/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/prices.csv",index=False)

## <font style = 'color:black'>003 Get HDB coordinates</font>
---
Get HDB postal code and geo coordinates using Onemap API.



In [11]:
## Function for getting postal code, geo coordinates of addresses

def get_address_info(unique_address, retries=3, sleep_sec=0.2):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": unique_address,
        "returnGeom": "Y",
        "getAddrDetails": "Y"
    }

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()

            if data.get("found", 0) > 0:
                result = data["results"][0]
                return {
                    "address": unique_address,
                    "postal_code": result.get("POSTAL"),
                    "latitude": float(result.get("LATITUDE")),
                    "longitude": float(result.get("LONGITUDE"))
                }
            else:
                return {
                    "address": unique_address,
                    "postal_code": None,
                    "latitude": None,
                    "longitude": None
                }

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(sleep_sec)
            else:
                print(f"Failed after retries: {unique_address}")
                return {
                    "address": unique_address,
                    "postal_code": None,
                    "latitude": None,
                    "longitude": None
                }


In [12]:
##Save progress EVERY 500 rows
#rows = []

#for i, addr in enumerate(tqdm(unique_address)):
#    rows.append(get_address_info(addr))

#    # Save checkpoint every 500 rows
#    if (i + 1) % 500 == 0:
#        temp_df = pd.DataFrame(rows)
#        temp_df.to_csv(
#            "/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_geocode_partial.csv",
#            index=False
#        )

#geo_df = pd.DataFrame(rows)


In [13]:
# to #out when done

#replace failed address
import os #input & output

# 1. Define your file path
file_path = '/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/prices.csv'

# 2. Create a "map" of replacements { 'OLD': 'NEW' }
replacements = {
   '670A EDGEFIELD PLAINS': '670A EDGEFIELD PLAINS WATERWAY RIDGES',
   '34 BEDOK STH AVE 2': '34 BEDOK SOUTH AVENUE 2',
   '357 YUNG AN RD': '357 YUNG AN ROAD',
   '119 SIMEI ST 1': '119 SIMEI STREET 1',
   '5 JLN BATU': '5 JLN BATU DI TANJONG RHU',
   '6 JLN BATU': '6 JLN BATU DI TANJONG RHU',
   '52 KENT RD': '52 KENT RD KENT VILLE',
   '54 KENT RD': '54 KENT RD KENT VILLE'
}

# 3. Read the original content
if os.path.exists(file_path):
   with open(file_path, 'r') as f:
       content = f.read()

   # 4. Loop through the dictionary and apply each replacement
   for old_word, new_word in replacements.items():
       content = content.replace(old_word, new_word)

   # 5. Write the final version back to the file
   with open(file_path, 'w') as f:
       f.write(content)

   print(f"Success! Replaced {len(replacements)} unique strings.")
else:
   print("File not found. Please check the path.")

Success! Replaced 8 unique strings.


In [14]:
# to #out when done

#resave progress EVERY 500 rows after replacing failed address
rows = []

for i, addr in enumerate(tqdm(unique_address)):
   rows.append(get_address_info(addr))

   # Save checkpoint every 500 rows
   if (i + 1) % 500 == 0:
       temp_df = pd.DataFrame(rows)
       temp_df.to_csv(
           "/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_geocode_partial.csv",
           index=False
       )

geo_df = pd.DataFrame(rows)


100%|██████████| 9697/9697 [1:53:22<00:00,  1.43it/s]


In [15]:
# to #out when done

geo_df.head()

,address,postal_code,latitude,longitude
0,566 HOUGANG ST 51,530566,1.381272,103.888435
1,241 KIM KEAT LINK,310241,1.330505,103.855776
2,858 TAMPINES AVE 5,520858,1.354293,103.937595
3,38C BENDEMEER RD,333038,1.321120,103.866787
4,504D YISHUN ST 51,764504,1.418334,103.843722


In [16]:
# to #out when done

geo_df.shape

(9697, 4)

In [17]:
# to #out when done

geo_df.to_csv(
   "/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_address_geocode.csv",
   index=False
)

print("CSV saved successfully")

CSV saved successfully


In [18]:
# find_postal(unique_address, 'Data/flat_coordinates')
flat_coord = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_address_geocode.csv')
flat_coord = flat_coord[['address','latitude','longitude']]
flat_coord.head()

,address,latitude,longitude
0,566 HOUGANG ST 51,1.381272,103.888435
1,241 KIM KEAT LINK,1.330505,103.855776
2,858 TAMPINES AVE 5,1.354293,103.937595
3,38C BENDEMEER RD,1.321120,103.866787
4,504D YISHUN ST 51,1.418334,103.843722


In [19]:
## Function for getting closest distance of each location from a list of amenities location

from geopy.distance import geodesic

def find_nearest(house, amenity, radius=2):
    """
    this function finds the nearest locations from the 2nd table from the 1st address
    Both are dataframes with a specific format:
        1st column: any string column ie addresses taken from the "find_postal_address.py"
        2nd column: latitude (float)
        3rd column: longitude (float)
    Column name doesn't matter.
    It also finds the number of amenities within the given radius (default=2)
    """
    results = {}
    # first column must be address
    for index,flat in enumerate(house.iloc[:,0]):

        # 2nd column must be latitude, 3rd column must be longitude
        flat_loc = (house.iloc[index,1],house.iloc[index,2])
        flat_amenity = ['','',100,0]
        for ind, eachloc in enumerate(amenity.iloc[:,0]):
            amenity_loc = (amenity.iloc[ind,1],amenity.iloc[ind,2])
            distance = geodesic(flat_loc,amenity_loc)
            distance = float(str(distance)[:-3]) # convert to float

            if distance <= radius:   # compute number of amenities in 2km radius
                flat_amenity[3] += 1

            if distance < flat_amenity[2]: # find nearest amenity
                flat_amenity[0] = flat
                flat_amenity[1] = eachloc
                flat_amenity[2] = distance

        results[flat] = flat_amenity
    return results

In [20]:
def dist_from_location(house, location):
    """
    this function finds the distance of a location from the 1st address
    First is a dataframe with a specific format:
        1st column: any string column ie addresses taken from the "find_postal_address.py"
        2nd column: latitude (float)
        3rd column: longitude (float)
    Column name doesn't matter.
    Second is tuple with latitude and longitude of location
    """
    results = {}
    # first column must be address
    for index,flat in enumerate(house.iloc[:,0]):

        # 2nd column must be latitude, 3rd column must be longitude
        flat_loc = (house.iloc[index,1],house.iloc[index,2])
        flat_amenity = ['',100]
        distance = geodesic(flat_loc,location)
        distance = float(str(distance)[:-3]) # convert to float
        flat_amenity[0] = flat
        flat_amenity[1] = distance
        results[flat] = flat_amenity
    return results

## <font style = 'color:black'>004 Get School coordinates</font>
---
1. Load all School CSV.
2. Get School geo coordinates using Onemap API.
3. For each HDB, find:
  *   Nearest School
  *   Nearest School Dist
  *   Number of school <2km





In [21]:
school = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/Main CSV/Generalinformationofschools.csv', encoding='cp1252')
school.head(10)

,school_name,url_address,address,postal_code,telephone_no,telephone_no_2,fax_no,fax_no_2,email_address,mrt_desc,...,nature_code,session_code,mainlevel_code,sap_ind,autonomous_ind,gifted_ind,ip_ind,mothertongue1_code,mothertongue2_code,mothertongue3_code
0,ADMIRALTY PRIMARY SCHOOL,https://admiraltypri.moe.edu.sg/,11 WOODLANDS CIRCLE,738907,63620598,na,63627512,na,ADMIRALTY_PS@MOE.EDU.SG,Admiralty Station,...,CO-ED SCHOOL,FULL DAY,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
1,ADMIRALTY SECONDARY SCHOOL,http://www.admiraltysec.moe.edu.sg,31 WOODLANDS CRESCENT,737916,63651733,63654596,63652774,na,Admiralty_SS@moe.edu.sg,ADMIRALTY MRT,...,CO-ED SCHOOL,SINGLE SESSION,SECONDARY (S1-S5),No,No,No,No,CHINESE,MALAY,TAMIL
2,AHMAD IBRAHIM PRIMARY SCHOOL,http://www.ahmadibrahimpri.moe.edu.sg,10 YISHUN STREET 11,768643,67592906,na,67592927,na,AIPS@MOE.EDU.SG,Yishun,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
3,AHMAD IBRAHIM SECONDARY SCHOOL,http://www.ahmadibrahimsec.moe.edu.sg,751 YISHUN AVENUE 7,768928,67585384,na,67557778,na,AISS@MOE.EDU.SG,"CANBERRA MRT, YISHUN MRT",...,CO-ED SCHOOL,SINGLE SESSION,SECONDARY (S1-S5),No,No,No,No,CHINESE,MALAY,TAMIL
4,AI TONG SCHOOL,http://www.aitong.moe.edu.sg,100 Bright Hill Drive,579646,64547672,na,64532726,na,AITONG_SCH@MOE.EDU.SG,Bishan MRT,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,Yes,No,No,No,CHINESE,na,na
5,ALEXANDRA PRIMARY SCHOOL,http://alexandrapri.moe.edu.sg,2A Prince Charles Crescent,159016,62485400,na,62485409,na,alexandra_ps@moe.edu.sg,Redhill Station; Tiong Bahru Station,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
6,ANCHOR GREEN PRIMARY SCHOOL,http://www.anchorgreenpri.moe.edu.sg,31 Anchorvale Drive,544969,68861356,na,63159825,na,anchorgreen_ps@moe.edu.sg,MRT : NE16-Sengkang; LRT : SW7-TongKang,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
7,ANDERSON PRIMARY SCHOOL,http://www.andersonpri.moe.edu.sg,19 ANG MO KIO AVE 9,569785,64560340,na,65522310,na,ANDERSON_PS@MOE.EDU.SG,Yio Chu Kang MRT Station (NS 15); Lentor MRT S...,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
8,ANDERSON SECONDARY SCHOOL,http://www.andersonsec.moe.edu.sg,10 ANG MO KIO STREET 53,569206,64598303,na,64586104,na,ANDERSON_SS@MOE.EDU.SG,"ANG MO KIO MRT, YIO CHU KANG MRT",...,CO-ED SCHOOL,SINGLE SESSION,SECONDARY (S1-S5),No,Yes,No,No,CHINESE,MALAY,TAMIL
9,ANDERSON SERANGOON JUNIOR COLLEGE,www.asrjc.moe.edu.sg,1033 Upper Serangoon Road,534768,64596822,na,64598734,na,asrjc@moe.edu.sg,Kovan MRT Station,...,CO-ED SCHOOL,FULL DAY,JUNIOR COLLEGE,No,No,No,No,CHINESE,MALAY,TAMIL


In [22]:
# Check the columns and rows
print(f'School shape: {school.shape}')
print(f'School data types:\n{school.dtypes}\n')

School shape: (337, 31)
School data types:
school_name           object
url_address           object
address               object
postal_code            int64
telephone_no          object
telephone_no_2        object
fax_no                object
fax_no_2              object
email_address         object
mrt_desc              object
bus_desc              object
principal_name        object
first_vp_name         object
second_vp_name        object
third_vp_name         object
fourth_vp_name        object
fifth_vp_name         object
sixth_vp_name         object
dgp_code              object
zone_code             object
type_code             object
nature_code           object
session_code          object
mainlevel_code        object
sap_ind               object
autonomous_ind        object
gifted_ind            object
ip_ind                object
mothertongue1_code    object
mothertongue2_code    object
mothertongue3_code    object
dtype: object



In [23]:
school_name = list(school['school_name'])
unique_school_name = list(set(school_name))
school_postal = list(school['postal_code'])
unique_school_postal = list(set(school_postal))
school_address= list(school['address'])
unique_school_address = list(set(school_address))

print('Unique schools:', len(unique_school_name))
print('Unique postals:', len(unique_school_postal))
print('Unique addresses:', len(unique_school_address))

Unique schools: 337
Unique postals: 335
Unique addresses: 336


In [24]:
## Function for getting postal code, geo coordinates of schools

def get_school_info(search_val, retries=3, sleep_sec=0.2):
    """
    Fetches specific geo-coordinates for a single search value.
    """
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y"
    }

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()

            # Check if OneMap found a result
            if data.get("found", 0) > 0:
                result = data["results"][0]
                # Return ONLY the data for this specific result
                return {
                    "School address": result.get("ADDRESS"),
                    "School postal": result.get("POSTAL"),
                    "School latitude": float(result.get("LATITUDE")),
                    "School longitude": float(result.get("LONGITUDE"))
                }
            else:
                # If no result found, return the search term and None for coords
                return {
                    "School address": search_val,
                    "School postal": None,
                    "School latitude": None,
                    "School longitude": None
                }

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(sleep_sec)
            else:
                print(f"Failed to fetch data for: {search_val}")
                return {
                    "School address": search_val,
                    "School postal": None,
                    "School latitude": None,
                    "School longitude": None
                }


In [25]:
rows = []

# Loop through both the name and address at the same time
for name, addr in tqdm(zip(school['school_name'], school['address']), total=len(school)):
    result = get_school_info(addr)

    # Add the school name to the dictionary
    result['School name'] = name
    # Ensure the address stays as the clean street name
    result['School address'] = addr

    rows.append(result)

# Create the final DataFrame
schgeo_df = pd.DataFrame(rows)

# Reorder columns to put 'School name' first
cols = ['School name', 'School address', 'School postal', 'School latitude', 'School longitude']
schgeo_df = schgeo_df[cols]

schgeo_df.head()

# Check the results - each cell should now have ONE value
print(schgeo_df.head())

100%|██████████| 337/337 [03:28<00:00,  1.62it/s]

                      School name            School address School postal  \
0        ADMIRALTY PRIMARY SCHOOL    11 WOODLANDS CIRCLE           738907   
1      ADMIRALTY SECONDARY SCHOOL  31 WOODLANDS CRESCENT           737916   
2    AHMAD IBRAHIM PRIMARY SCHOOL    10 YISHUN STREET 11           768643   
3  AHMAD IBRAHIM SECONDARY SCHOOL    751 YISHUN AVENUE 7           768928   
4                  AI TONG SCHOOL  100 Bright Hill Drive           579646   

   School latitude  School longitude  
0         1.442635        103.800040  
1         1.445891        103.802398  
2         1.433153        103.832942  
3         1.436060        103.829719  
4         1.360583        103.833020  


In [26]:
# Identify rows where latitude OR longitude is missing (NaN)
missinggeo_schools = schgeo_df[schgeo_df['School latitude'].isna() | schgeo_df['School longitude'].isna()]

# Display the schools that are about to be removed
print(f"Found {len(missinggeo_schools)} schools with missing coordinates:")
display(missinggeo_schools)

Found 1 schools with missing coordinates:


,School name,School address,School postal,School latitude,School longitude
244,"SCHOOL OF THE ARTS, SINGAPORE",1 Zubir Said Drive 05 01,None,NaN,NaN


Ok to drop SCHOOL OF THE ARTS subsequently


In [27]:
# Check the columns and rows
print(f'School Geo shape: {schgeo_df.shape}')
print(f'School Geo data types:\n{schgeo_df.dtypes}\n')
schgeo_df.head()

School Geo shape: (337, 5)
School Geo data types:
School name          object
School address       object
School postal        object
School latitude     float64
School longitude    float64
dtype: object



,School name,School address,School postal,School latitude,School longitude
0,ADMIRALTY PRIMARY SCHOOL,11 WOODLANDS CIRCLE,738907,1.442635,103.800040
1,ADMIRALTY SECONDARY SCHOOL,31 WOODLANDS CRESCENT,737916,1.445891,103.802398
2,AHMAD IBRAHIM PRIMARY SCHOOL,10 YISHUN STREET 11,768643,1.433153,103.832942
3,AHMAD IBRAHIM SECONDARY SCHOOL,751 YISHUN AVENUE 7,768928,1.436060,103.829719
4,AI TONG SCHOOL,100 Bright Hill Drive,579646,1.360583,103.833020


In [28]:
schgeo_df.to_csv(
   "/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/school_geocode.csv",
   index=False
)

print("CSV saved successfully")

CSV saved successfully


In [29]:
# 1. Load data
hdb = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_address_geocode.csv')
schools = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/school_geocode.csv')
hdb_initial_count = len(hdb)
schools_initial_count = len(schools)


In [30]:
from sklearn.neighbors import BallTree

# 2. Pre-processing: Remove rows with missing coordinates to avoid errors
hdb = hdb.dropna(subset=['latitude', 'longitude'])
schools = schools.dropna(subset=['School latitude', 'School longitude'])
hdb_dropped_count = hdb_initial_count - len(hdb)
print(f"Removed {hdb_dropped_count} rows from the 'hdb' dataset due to missing coordinates.")
schools_dropped_count = schools_initial_count - len(schools)
print(f"Removed {schools_dropped_count} rows from the 'schools' dataset due to missing coordinates.")

def find_nearest(flats_df, schools_df):
   # Convert latitude/longitude to radians for the Haversine formula
   flat_coords = np.radians(flats_df[['latitude', 'longitude']])
   school_coords = np.radians(schools_df[['School latitude', 'School longitude']])

   earth_radius = 6371.0 # Radius of Earth in KM

   # Build the search tree using schools as the targets
   tree = BallTree(school_coords, metric='haversine')

   # Query 1: Find the single nearest school (k=1)
   dist, ind = tree.query(flat_coords, k=1)

    # Query 2: Count schools within a 1km radius
   # (1 / 6371) converts 1km into radians for the tree
   count_1km = tree.query_radius(flat_coords, r=1.0/earth_radius, count_only=True)

   # Query 3: Count schools within a 2km radius
   # (2 / 6371) converts 2km into radians for the tree
   count_2km = tree.query_radius(flat_coords, r=2.0/earth_radius, count_only=True)

   # Construct the final result table
   res = pd.DataFrame({
       'flat address': flats_df['address'].values,
       'Nearest school': schools_df['School name'].iloc[ind.flatten()].values,
       'school_dist': (dist.flatten() * earth_radius),
       'num_school_1km': count_1km,
       'num_school_2km': count_2km
   })

   return res

# 3. Execute the function
flat_school = find_nearest(hdb, schools)

# 4. View and save the results
print(flat_school.head())

Removed 0 rows from the 'hdb' dataset due to missing coordinates.
Removed 1 rows from the 'schools' dataset due to missing coordinates.
         flat address                Nearest school  school_dist  \
0   566 HOUGANG ST 51      PALM VIEW PRIMARY SCHOOL     0.427093   
1   241 KIM KEAT LINK        PEI CHUN PUBLIC SCHOOL     0.699232   
2  858 TAMPINES AVE 5      JUNYUAN SECONDARY SCHOOL     0.419164   
3    38C BENDEMEER RD      BENDEMEER PRIMARY SCHOOL     0.171904   
4   504D YISHUN ST 51  NORTHBROOKS SECONDARY SCHOOL     0.317984   

   num_school_1km  num_school_2km  
0               8              31  
1               4              16  
2               8              15  
3               2              12  
4               6              15  


In [31]:
flat_school.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_school_proximity.csv", index=False)
print("CSV saved successfully")

CSV saved successfully


## <font style = 'color:black'>005 Top 27 Primary schools</font>
---

1. Extract Top 27 schools based on GEP offered and Top schools listed in [Primary School finder's ranking](https://www.primaryschoolfinder.sg/school-ranking)
2. Get School geo coordinates using Onemap API.
3. For each HDB, find:
  *   Nearest School
  *   Nearest School Dist
  *   Number of school <1km & <2km


###**Feature Engineering - Education & Proximity Analysis**
**1. Understanding the P1 Registration Framework**
>Primary 1 (P1) admission in Singapore is a tiered process where priority is determined by a child's citizenship, prior affiliations, and residential proximity. This framework creates a measurable "Proximity Premium" in the HDB resale market.

**The Registration Phases**

>**Phase 1 (Siblings):**
Highest priority; guaranteed placement for children with older siblings in the school.

>**Phase 2A (Affiliations):** For children of alumni, staff, or those from an MOE Kindergarten located within the school.

>**Phase 2B (Community/Religious):**
For parent volunteers (40+ hours), endorsed church/clan members, or community leaders. 20 places reserved. Balloting occurs if applicants exceed 20 spots, generally prioritized by citizenship and distance (<1km, 1-2km, >2km)

>**Phase 2C (General Public):**
The "Open Phase" based purely on distance. 40 places reserved. Balloting occurs if applicants exceed 40 spots, generally prioritized by citizenship and distance (<1km, 1-2km, >2km)



**2. Strategic Update: The GEP Transition (2027)**
Starting in 2027, the centralized Gifted Education Programme (GEP), currently limited to 9 "top" schools, will be discontinued and replaced by a school-based approach for Higher-Ability Learners (HAL).

>**Current State:** 9 GEP schools drive extreme localized price premiums.

>**Future State:** High-ability programs will be available in all primary schools, potentially decentralizing the "Elite School" price effect.

**3. Feature Selection: High-Priority Schools**
For this model, we focus on 27 High-Priority Schools identified by their GEP status, and historical Phase 2b & 2C popularity.

**Technical Methodology**
We use a BallTree algorithm with the Haversine metric to calculate geospatial distances. We define two critical features for our model:

>**num_priority_sch_1km:** The number of elite schools within the highest-priority balloting zone.

>**num_priority_sch_2km:** The number of elite schools within the secondary balloting zone.

**4. The 30-Month Residency Rule**
To prevent "address renting," MOE requires families who gain entry via proximity to reside at the registered address for **at least 30 months**. This rule stabilizes the resale market by preventing high-frequency flipping of properties near popular schools.

In [32]:
# 1. Definepriority names list from previous setup
priority_names = [
    "TAO NAN SCHOOL", "AI TONG SCHOOL", "NANYANG PRIMARY SCHOOL",
    "HENRY PARK PRIMARY SCHOOL", "RAFFLES GIRLS' PRIMARY SCHOOL",
    "PEI HWA PRESBYTERIAN PRIMARY SCHOOL", "METHODIST GIRLS' SCHOOL (PRIMARY)",
    "NAN CHIAU PRIMARY SCHOOL", "CHIJ ST. NICHOLAS GIRLS' SCHOOL",
    "CHIJ PRIMARY (TOA PAYOH)", "RED SWASTIKA SCHOOL", "KONG HWA SCHOOL",
    "ANGLO-CHINESE SCHOOL (JUNIOR)", "MAHA BODHI SCHOOL", "HOLY INNOCENTS' PRIMARY SCHOOL",
    "ST. JOSEPH'S INSTITUTION JUNIOR", "CHONGFU SCHOOL", "NAN HUA PRIMARY SCHOOL",
    "ANGLO-CHINESE SCHOOL (PRIMARY)", "CATHOLIC HIGH SCHOOL",
    "MARIS STELLA HIGH SCHOOL", "ROSYTH SCHOOL", "PEI CHUN PUBLIC SCHOOL",
    "FAIRFIELD METHODIST SCHOOL (PRIMARY)", "RULANG PRIMARY SCHOOL",
    "NORTHLAND PRIMARY SCHOOL", "ST. HILDA'S PRIMARY SCHOOL"
]

# 2. Filter the Geo Dataframe
# We use .str.upper() to ensure the casing matches list
priority_geo_df = schgeo_df[schgeo_df['School name'].str.upper().isin(priority_names)].copy()

# 3. Verify the results
print(f"Extracted {priority_geo_df.shape[0]} schools out of {len(priority_names)}")
priority_geo_df.head(27)

Extracted 27 schools out of 27


,School name,School address,School postal,School latitude,School longitude
4,AI TONG SCHOOL,100 Bright Hill Drive,579646,1.360583,103.833020
16,ANGLO-CHINESE SCHOOL (JUNIOR),16 WINSTEDT ROAD,227988,1.309350,103.840950
17,ANGLO-CHINESE SCHOOL (PRIMARY),50 BARKER ROAD,309918,1.318371,103.835610
47,CATHOLIC HIGH SCHOOL,9 BISHAN STREET 22,579767,1.354525,103.844901
60,CHIJ PRIMARY (TOA PAYOH),628 Lorong 1 Toa Payoh,319765,1.332753,103.841847
63,CHIJ ST. NICHOLAS GIRLS' SCHOOL,501 ANG MO KIO STREET 13,569405,1.373477,103.834253
65,CHONGFU SCHOOL,170 YISHUN AVENUE 6,768959,1.438396,103.839309
98,FAIRFIELD METHODIST SCHOOL (PRIMARY),100 DOVER ROAD,139648,1.301004,103.785456
122,HENRY PARK PRIMARY SCHOOL,1 HOLLAND GROVE ROAD,278790,1.316676,103.784296
125,HOLY INNOCENTS' PRIMARY SCHOOL,5 Lorong Low Koon,536451,1.366938,103.894115


In [33]:
priority_geo_df.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/Top_primary_geocode.csv", index=False)

print("CSV saved successfully")


CSV saved successfully


In [34]:

# 1. Pre-processing
# priority_geo_df is ready (from previous filtering step)
# and hdb data is loaded
hdb = hdb.dropna(subset=['latitude', 'longitude'])
priority_schools = priority_geo_df.dropna(subset=['School latitude', 'School longitude'])

def find_nearest_priority(flats_df, schools_df):
    # Convert latitude/longitude to radians for the Haversine formula
    flat_coords = np.radians(flats_df[['latitude', 'longitude']])
    school_coords = np.radians(schools_df[['School latitude', 'School longitude']])

    earth_radius = 6371.0 # Radius of Earth in KM

    # Build the search tree using the 27 priority schools
    tree = BallTree(school_coords, metric='haversine')

    # Query 1: Find the single nearest priority school
    dist, ind = tree.query(flat_coords, k=1)

    # Query 2: Count priority schools within 1km radius
    count_1km = tree.query_radius(flat_coords, r=1.0/earth_radius, count_only=True)

    # Query 3: Count priority schools within 2km radius
    count_2km = tree.query_radius(flat_coords, r=2.0/earth_radius, count_only=True)

    # Construct the final result table
    res = pd.DataFrame({
        'flat address': flats_df['address'].values,
        'Nearest priority school': schools_df['School name'].iloc[ind.flatten()].values,
        'priority_school_dist_km': (dist.flatten() * earth_radius),
        'num_priority_sch_1km': count_1km,
        'num_priority_sch_2km': count_2km
    })

    return res

# 2. Execute using priority_geo_df
flat_priority_analysis = find_nearest_priority(hdb, priority_schools)

# 3. View the results
print(flat_priority_analysis.head())

         flat address     Nearest priority school  priority_school_dist_km  \
0   566 HOUGANG ST 51    NAN CHIAU PRIMARY SCHOOL                 1.254083   
1   241 KIM KEAT LINK      PEI CHUN PUBLIC SCHOOL                 0.699232   
2  858 TAMPINES AVE 5  ST. HILDA'S PRIMARY SCHOOL                 0.517018   
3    38C BENDEMEER RD      PEI CHUN PUBLIC SCHOOL                 2.156479   
4   504D YISHUN ST 51    NORTHLAND PRIMARY SCHOOL                 0.463772   

   num_priority_sch_1km  num_priority_sch_2km  
0                     0                     3  
1                     1                     3  
2                     1                     1  
3                     0                     0  
4                     1                     1  


In [35]:
flat_priority_analysis.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_top27school_proximity.csv", index=False)
print("CSV saved successfully")

CSV saved successfully


## <font style = 'color:black'>006 Get MRT Coordinates </font>
---

1. Load MRT excel from LTA's Public Transport page 2 [Train Station Codes and Chinese Names](https://datamall.lta.gov.sg/content/datamall/en/static-data.html#Public%20Transport).
2. Get MRT geo coordinates using Onemap API.
3. For each HDB, find:
  *   Nearest MRT
  *   Nearest MRT Dist
  *   Number of MRT <1km
  
  1km walking time:
*    Brisk walk = 10mins
*    Normal pace = 15 to 20 mins






Looking at HDB
MRT distance
MRT 1km
Pick CBD / interchange MRT 1km



In [36]:
file_path = '/content/drive/MyDrive/GA Data analytics Capstone/Main CSV/Train Station Codes and Chinese Names.xls' # Replace with your copied path
mrt = pd.read_excel(file_path)
mrt = mrt.drop(columns=['mrt_station_chinese', 'mrt_line_chinese'])
mrt = mrt.rename(columns={
    'stn_code': 'Station code',
    'mrt_station_english': 'Station Name',
    'mrt_line_english': 'MRT Line'})
mrt.head()

,Station code,Station Name,MRT Line
0,NS1,Jurong East,North-South Line
1,NS2,Bukit Batok,North-South Line
2,NS3,Bukit Gombak,North-South Line
3,NS4,Choa Chu Kang,North-South Line
4,NS5,Yew Tee,North-South Line


In [37]:
unique_mrtline = mrt['MRT Line'].unique()
print(unique_mrtline)

['North-South Line ' 'East-West Line' 'Changi Airport Branch Line'
 'North East Line' 'Circle Line' 'Circle Line Extension' 'Downtown Line'
 'Bukit Panjang LRT' 'Sengkang LRT' 'Punggol LRT'
 'Thomson-East Coast Line']


In [38]:
#Drop rows where "MRT Line" contains "LRT"
# We use case=False to be safe, though usually it's uppercase in the file
mrt = mrt[~mrt['MRT Line'].str.contains('LRT', case=False, na=False)]
unique_mrtline = mrt['MRT Line'].unique()
print(unique_mrtline)
mrt.head()

['North-South Line ' 'East-West Line' 'Changi Airport Branch Line'
 'North East Line' 'Circle Line' 'Circle Line Extension' 'Downtown Line'
 'Thomson-East Coast Line']


,Station code,Station Name,MRT Line
0,NS1,Jurong East,North-South Line
1,NS2,Bukit Batok,North-South Line
2,NS3,Bukit Gombak,North-South Line
3,NS4,Choa Chu Kang,North-South Line
4,NS5,Yew Tee,North-South Line


In [39]:
def get_coordinates(station_name):
    """Fetches Lat/Long from OneMap Search API."""
    url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={station_name} MRT STATION&returnGeom=Y&getAddrDetails=N"

    try:
        response = requests.get(url)
        data = response.json()

        if data['found'] > 0:
            result = data['results'][0]
            return result['LATITUDE'], result['LONGITUDE']
        else:
            return None, None
    except Exception as e:
        print(f"Error fetching {station_name}: {e}")
        return None, None

# Processing the data
results = []
# Iterate over DataFrame rows using iterrows()
for index, row in mrt.iterrows():
    station_name = row["Station Name"]
    lat, lon = get_coordinates(station_name)

    # Create a dictionary for the current station's data
    station_data = {
        "Station code": row["Station code"],
        "Station Name": row["Station Name"],
        "MRT Line": row["MRT Line"],
        "Latitude": lat,
        "Longitude": lon
    }
    results.append(station_data)
    # Slight delay to avoid hitting API rate limits
    time.sleep(0.1)

# Display as a DataFrame
mrtgeo = pd.DataFrame(results)
print(mrtgeo[['Station code', 'Station Name', 'MRT Line', 'Latitude', 'Longitude']])


    Station code    Station Name                 MRT Line          Latitude  \
0            NS1     Jurong East        North-South Line    1.3330275154394   
1            NS2     Bukit Batok        North-South Line   1.34903331201636   
2            NS3    Bukit Gombak        North-South Line   1.35861159094192   
3            NS4   Choa Chu Kang        North-South Line   1.38536846992147   
4            NS5         Yew Tee        North-South Line   1.39753506936297   
..           ...             ...                      ...               ...   
166         TE25  Tanjong Katong  Thomson-East Coast Line  1.29931925769899   
167         TE26   Marine Parade  Thomson-East Coast Line  1.30261163483383   
168         TE27  Marine Terrace  Thomson-East Coast Line  1.30666510344814   
169         TE28          Siglap  Thomson-East Coast Line  1.31000869432289   
170         TE29        Bayshore  Thomson-East Coast Line  1.31312248291264   

            Longitude  
0    103.742370487607  
1  

In [40]:
#Save to CSV
mrtgeo.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/mrt_coordinates.csv", index=False)

In [41]:
# 1. Load data
hdb_df = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_address_geocode.csv')
mrt_df = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/mrt_coordinates.csv')
hdb_df_initial_count = len(hdb_df)
mrt_df_initial_count = len(mrt_df)


In [42]:
# 2. Pre-processing: Remove rows with missing coordinates to avoid errors
hdb_df = hdb_df.dropna(subset=['latitude', 'longitude'])
mrt_df = mrt_df.dropna(subset=['Latitude', 'Longitude'])
hdb_df_dropped_count = hdb_df_initial_count - len(hdb_df)
print(f"Removed {hdb_df_dropped_count} rows from the 'hdb_df' dataset due to missing coordinates.")
mrt_df_dropped_count = mrt_df_initial_count - len(mrt_df)
print(f"Removed {mrt_df_dropped_count} rows from the 'mrt_df' dataset due to missing coordinates.")

# Use unique MRT station locations (to avoid double-counting interchanges)
mrt_unique = mrt_df[['Station Name', 'Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)

# Haversine Formula (Vectorized for speed)
def haversine_dist(lat1, lon1, lat2, lon2):
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Differences
    dlat = lat2[:, np.newaxis] - lat1
    dlon = lon2[:, np.newaxis] - lon1

    # Formula
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2[:, np.newaxis]) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c # Radius of earth in km
    return km

# Prepare coordinates for calculation
hdb_lats = hdb_df['latitude'].values
hdb_lons = hdb_df['longitude'].values
mrt_lats = mrt_unique['Latitude'].values
mrt_lons = mrt_unique['Longitude'].values

# Calculate a matrix of distances (HDB rows x MRT columns)
# This finds the distance from every HDB to every MRT
dist_matrix = haversine_dist(hdb_lats, hdb_lons, mrt_lats, mrt_lons)

# Extract required insights
# Find index of the minimum distance for each HDB
nearest_idx = np.argmin(dist_matrix, axis=0)

hdb_df['Nearest MRT Station'] = mrt_unique.iloc[nearest_idx]['Station Name'].values
hdb_df['MRT_dist_km'] = np.min(dist_matrix, axis=0)
hdb_df['num_mrt_1km'] = np.sum(dist_matrix <= 1.0, axis=0)

# Final Formatting
result_df = hdb_df[['address', 'Nearest MRT Station', 'MRT_dist_km', 'num_mrt_1km']]
result_df = result_df.rename(columns={'address': 'flat address'})

# Save to CSV
result_df.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_mrt_proximity.csv", index=False)
print(result_df.head())

Removed 0 rows from the 'hdb_df' dataset due to missing coordinates.
Removed 0 rows from the 'mrt_df' dataset due to missing coordinates.
         flat address Nearest MRT Station  MRT_dist_km  num_mrt_1km
0   566 HOUGANG ST 51            Buangkok     0.579381            1
1   241 KIM KEAT LINK           Toa Payoh     0.949634            1
2  858 TAMPINES AVE 5       Tampines West     0.980549            1
3    38C BENDEMEER RD       Geylang Bahru     0.537944            3
4   504D YISHUN ST 51              Khatib     1.198728            0


In [44]:
# Load the MRT coordinates
mrt_df = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/mrt_coordinates.csv')

# Clean station and line names
mrt_df['Station Name'] = mrt_df['Station Name'].str.strip()
mrt_df['MRT Line'] = mrt_df['MRT Line'].str.strip()

# Group to find interchanges
interchange_counts = mrt_df.groupby('Station Name')['MRT Line'].nunique().reset_index()
interchange_counts.columns = ['Station Name', 'number of lines']

# Filter for interchanges (more than 1 line)
interchanges = interchange_counts[interchange_counts['number of lines'] > 1]

# Get the lines served for each station
lines_served = mrt_df.groupby('Station Name')['MRT Line'].unique().apply(lambda x: ', '.join(sorted(x))).reset_index()
lines_served.columns = ['Station Name', 'lines served']

# Get coordinates for each station.
# Since an interchange might have multiple rows (one for each line),
# we take the first coordinate entry for that station name.
coords = mrt_df.groupby('Station Name')[['Latitude', 'Longitude']].first().reset_index()

# Merge all information
final_interchange_list = interchanges.merge(lines_served, on='Station Name').merge(coords, on='Station Name')

# Sort by number of lines (descending) and then name
final_interchange_list = final_interchange_list.sort_values(by=['number of lines', 'Station Name'], ascending=[False, True])

# Select and order columns
final_interchange_list = final_interchange_list[['Station Name', 'number of lines', 'lines served', 'Latitude', 'Longitude']]

# Display and save
print(final_interchange_list)
final_interchange_list.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/mrtinterchange.csv", index=False)

       Station Name  number of lines  \
8       Dhoby Ghaut                3   
14       Marina Bay                3   
17      Outram Park                3   
0          Bayfront                2   
1            Bishan                2   
2   Botanic Gardens                2   
3             Bugis                2   
4       Buona Vista                2   
5         Caldecott                2   
6         Chinatown                2   
7         City Hall                2   
9              Expo                2   
10     HarbourFront                2   
11      Jurong East                2   
12     Little India                2   
13       MacPherson                2   
15           Newton                2   
16          Orchard                2   
18       Paya Lebar                2   
19        Promenade                2   
20    Raffles Place                2   
21        Serangoon                2   
22          Stevens                2   
23         Tampines                2   


In [45]:
# 1. Load the datasets
mrt_df = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/mrt_coordinates.csv')
hdb_df = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_address_geocode.csv')

# 2. Identify MRT Interchanges
# Clean names and find stations serving more than one unique line
mrt_df['Station Name'] = mrt_df['Station Name'].str.strip()
mrt_df['MRT Line'] = mrt_df['MRT Line'].str.strip()

interchange_counts = mrt_df.groupby('Station Name')['MRT Line'].nunique().reset_index()
interchange_names = interchange_counts[interchange_counts['MRT Line'] > 1]['Station Name']

# Extract unique coordinates for these interchanges
interchange_coords = mrt_df[mrt_df['Station Name'].isin(interchange_names)].groupby('Station Name')[['Latitude', 'Longitude']].first().reset_index()

# 3. Vectorized Haversine Formula for Distance Calculation
def haversine_dist(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2[:, np.newaxis] - lat1
    dlon = lon2[:, np.newaxis] - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2[:, np.newaxis]) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c # Earth radius in km

# 4. Perform Calculations
hdb_lats, hdb_lons = hdb_df['latitude'].values, hdb_df['longitude'].values
int_lats, int_lons = interchange_coords['Latitude'].values, interchange_coords['Longitude'].values

# Compute a distance matrix (HDB rows x Interchange columns)
dist_matrix = haversine_dist(hdb_lats, hdb_lons, int_lats, int_lons)

# Identify Nearest Interchange and count those within 1km
nearest_idx = np.argmin(dist_matrix, axis=0)
hdb_df['Nearest MRT interchange'] = interchange_coords.iloc[nearest_idx]['Station Name'].values
hdb_df['MRT_interchange_dist_km'] = np.min(dist_matrix, axis=0)
hdb_df['num_mrt_interchange_1km'] = np.sum(dist_matrix <= 1.0, axis=0)

# 5. Final Table Formatting
result_df = hdb_df[['address', 'Nearest MRT interchange', 'MRT_interchange_dist_km', 'num_mrt_interchange_1km']]
result_df = result_df.rename(columns={'address': 'flat address'})

# Save to CSV
result_df.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_mrtint_proximity.csv", index=False)
print(result_df.head())

         flat address Nearest MRT interchange  MRT_interchange_dist_km  \
0   566 HOUGANG ST 51               Serangoon                 3.826044   
1   241 KIM KEAT LINK               Caldecott                 1.958281   
2  858 TAMPINES AVE 5                Tampines                 1.905899   
3    38C BENDEMEER RD            Little India                 2.435009   
4   504D YISHUN ST 51               Woodlands                 6.730510   

   num_mrt_interchange_1km  
0                        0  
1                        0  
2                        0  
3                        0  
4                        0  


## <font style = 'color:black'>007 Get Bus stop, Hawker center coordinates </font>
---

1. Load API for [Bus stop coordinates](https://data.gov.sg/datasets/d_3f172c6feb3f4f92a2f47d93eed2908a/view).

2. Load API for [Hawker center coordinates](https://data.gov.sg/datasets?query=Hawker+Centres&resultId=d_4a086da0a5553be1d89383cd90d07ecd).

3. Load API for [Parks coordinates](https://data.gov.sg/datasets/d_0542d48f0991541706b58059381a6eca/view).

4. Load CSV for [Shopping coordinates](https://www.kaggle.com/datasets/karthikgangula/shopping-mall-coordinates).

In [46]:
# Define API Key
API_KEY = "x"

# Add headers
headers = {
    "x-api-key": API_KEY}

In [47]:
dataset_id = "d_3f172c6feb3f4f92a2f47d93eed2908a"
url = "https://api-open.data.gov.sg/v1/public/api/datasets/" + dataset_id + "/poll-download"


response = requests.get(url, headers=headers)
json_data = response.json()
if json_data['code'] != 0:
    print(json_data['errorMsg']) # Corrected key from 'errMsg' to 'errorMsg'
    exit(1)

url = json_data['data']['url']
response = requests.get(url)
print(response.text)

Output hidden; open in https://colab.research.google.com to view.

In [48]:
busstop_data = response.json()

# 1. Extract the features list
busstop_features = busstop_data['features']

# 2. Use a list comprehension to pull only the specific fields required
rows = []
for f in busstop_features:
    rows.append({
        'OBJECTID_1': f['properties']['OBJECTID_1'],
        'BUS_STOP_NUM': f['properties']['BUS_STOP_NUM'],
        # GeoJSON coordinates are [Longitude, Latitude]
        'longitude': f['geometry']['coordinates'][0],
        'latitude': f['geometry']['coordinates'][1]
    })

# 3. Create DataFrame and export
busstop = pd.DataFrame(rows)
busstop.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/bus_stops.csv", index=False)

print("CSV created successfully with the following columns:")
print(busstop.columns.tolist())
print(busstop.head())

CSV created successfully with the following columns:
['OBJECTID_1', 'BUS_STOP_NUM', 'longitude', 'latitude']
   OBJECTID_1 BUS_STOP_NUM   longitude  latitude
0      144151        58039  103.825582  1.446508
1      144152        58031  103.825139  1.446489
2      144153        14199  103.812726  1.281300
3      144154        22259  103.702624  1.340913
4      144155        66581  103.879735  1.382448


In [49]:
 #Hawker API request
dataset_id = "d_4a086da0a5553be1d89383cd90d07ecd"
url = "https://api-open.data.gov.sg/v1/public/api/datasets/" + dataset_id + "/poll-download"


response = requests.get(url,headers=headers)
json_data = response.json()
if json_data['code'] != 0:
    print(json_data['errMsg'])
    exit(1)

url = json_data['data']['url']
response = requests.get(url)
print(response.text)

{
	"type" : "FeatureCollection",
	"name" : "SSOT_HAWKERCENTRES",
	"features" : [
		{
			"type" : "Feature",
			"geometry" : {
				"type" : "Point",
				"coordinates" : [ 103.84661927383159, 1.2792312094873002 ]
			},
			"properties" : {
				"OBJECTID" : 119091,
				"LANDXADDRESSPOINT" : 29433.14995771,
				"LANDYADDRESSPOINT" : 29089.97730924,
				"ADDRESSBUILDINGNAME" : "MND COMPLEX (ANNEXE B)",
				"ADDRESSPOSTALCODE" : "069111",
				"ADDRESSSTREETNAME" : "MAXWELL ROAD",
				"DESCRIPTION" : "HUP Standard Upgrading",
				"NAME" : "Amoy Street Food Centre (Telok Ayer Food Centre)",
				"PHOTOURL" : "http://www.nea.gov.sg/images/default-source/Hawker-Centres-Division/resize_1265968760190.jpg",
				"ADDRESSBLOCKHOUSENUMBER" : "7",
				"STATUS" : "Existing",
				"AWARDED_DATE" : null,
				"IMPLEMENTATION_DATE" : null,
				"INFO_ON_CO_LOCATORS" : null,
				"ADDRESS_MYENV" : "National Development Building, Annex B, Telok Ayer Street, Singapore 069111",
				"EST_ORIGINAL_COMPLETION_DATE" : "

In [50]:
hawker_data = response.json()

# 1. Extract the features list
hawker_features = hawker_data['features']

# 2. Use a list comprehension to pull only the specific fields required
rows = []
for f in hawker_features:
    rows.append({
        'OBJECTID': f['properties']['OBJECTID'],
        'ADDRESSBUILDINGNAME': f['properties']['ADDRESSBUILDINGNAME'],
        # GeoJSON coordinates are [Longitude, Latitude]
        'longitude': f['geometry']['coordinates'][0],
        'latitude': f['geometry']['coordinates'][1]
    })

# 3. Create DataFrame and export
hawker = pd.DataFrame(rows)
hawker.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hawker_centres.csv", index=False)

print("CSV created successfully with the following columns:")
print(hawker.columns.tolist())
print(hawker.head())

CSV created successfully with the following columns:
['OBJECTID', 'ADDRESSBUILDINGNAME', 'longitude', 'latitude']
   OBJECTID     ADDRESSBUILDINGNAME   longitude  latitude
0    119091  MND COMPLEX (ANNEXE B)  103.846619  1.279231
1    119092   SKYRESIDENCE @ DAWSON  103.804694  1.297487
2    119093      PUNGGOL COAST MALL  103.908543  1.414518
3    119094    TELOK BLANGAH BEACON  103.808429  1.273250
4    119095      BUKIT TIMAH MARKET  103.776330  1.339237


In [51]:
#Parks API request

dataset_id = "d_0542d48f0991541706b58059381a6eca"
url = "https://api-open.data.gov.sg/v1/public/api/datasets/" + dataset_id + "/poll-download"

response = requests.get(url,headers=headers)
json_data = response.json()
if json_data['code'] != 0:
    print(json_data['errMsg'])
    exit(1)

url = json_data['data']['url']
response = requests.get(url)
print(response.text)

{
	"type" : "FeatureCollection",
	"name" : "NATIONALPARKS",
	"features" : [
		{
			"type" : "Feature",
			"geometry" : {
				"type" : "Point",
				"coordinates" : [ 103.83948703941307, 1.4280122121948617 ]
			},
			"properties" : {
				"OBJECTID" : 67581,
				"NAME" : "YISHUN POND PK",
				"X" : 28686.45805,
				"Y" : 45527.85272381,
				"INC_CRC" : "267641F12FE6C5B8",
				"FMEL_UPD_D" : "20260115000356"
			}
		},
		{
			"type" : "Feature",
			"geometry" : {
				"type" : "Point",
				"coordinates" : [ 103.83885324901108, 1.3872785047325376 ]
			},
			"properties" : {
				"OBJECTID" : 67582,
				"NAME" : "BANYAN VILLAS PG",
				"X" : 28615.93733365,
				"Y" : 41023.72504999,
				"INC_CRC" : "433066A008AE776B",
				"FMEL_UPD_D" : "20260115000356"
			}
		},
		{
			"type" : "Feature",
			"geometry" : {
				"type" : "Point",
				"coordinates" : [ 103.8384079546499, 1.390187591950426 ]
			},
			"properties" : {
				"OBJECTID" : 67583,
				"NAME" : "COUNTRYSIDE PG",
				"X" : 28566.38108947,


In [52]:
park_data = response.json()

# 1. Extract the features list
park_features = park_data['features']

# 2. Use a list comprehension to pull only the specific fields required
rows = []
for f in park_features:
    rows.append({
        'OBJECTID': f['properties']['OBJECTID'],
        'NAME': f['properties']['NAME'],
        # GeoJSON coordinates are [Longitude, Latitude]
        'longitude': f['geometry']['coordinates'][0],
        'latitude': f['geometry']['coordinates'][1]
    })

# 3. Create DataFrame and export
park = pd.DataFrame(rows)
park.to_csv("/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/parks.csv", index=False)

print("CSV created successfully with the following columns:")
print(park.columns.tolist())
print(park.head())

CSV created successfully with the following columns:
['OBJECTID', 'NAME', 'longitude', 'latitude']
   OBJECTID              NAME   longitude  latitude
0     67581    YISHUN POND PK  103.839487  1.428012
1     67582  BANYAN VILLAS PG  103.838853  1.387279
2     67583    COUNTRYSIDE PG  103.838408  1.390188
3     67584   KAMPONG GLAM PK  103.862996  1.302445
4     67585     THOMSON RD PG  103.843709  1.316260


In [53]:
# Load HDB geocode data
hdb_df = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_address_geocode.csv')

# Load amenity datasets (Ensure these files are uploaded to Colab)
# Standardize column names to 'latitude' and 'longitude' for each
bus_stops = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/bus_stops.csv')
hawkers = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hawker_centres.csv')
parks = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/parks.csv')
malls = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/Main CSV/shopping_mall_coordinates.csv')

# Rename columns for malls to match expected format
malls = malls.rename(columns={'LATITUDE': 'latitude', 'LONGITUDE': 'longitude'})

def calculate_amenity_metrics(hdb_df, amenity_df, dist_col, count_col):
    """Calculates nearest distance and count within 1km for a given amenity."""
    hdb_lats, hdb_lons = hdb_df['latitude'].values, hdb_df['longitude'].values
    am_lats, am_lons = amenity_df['latitude'].values, amenity_df['longitude'].values

    # Vectorized Haversine
    lat1, lon1, lat2, lon2 = map(np.radians, [hdb_lats, hdb_lons, am_lats, am_lons])

    # We compute distances in chunks if memory is an issue
    # For smaller datasets like Malls/Hawkers, we can do it in one go:
    dlat = lat2[:, np.newaxis] - lat1
    dlon = lon2[:, np.newaxis] - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2[:, np.newaxis]) * np.sin(dlon/2.0)**2
    dist_matrix = 6371 * 2 * np.arcsin(np.sqrt(a))

    hdb_df[dist_col] = np.min(dist_matrix, axis=0)
    hdb_df[count_col] = np.sum(dist_matrix <= 1.0, axis=0)
    return hdb_df

# Apply calculations for each amenity
print("Processing Bus Stops...")
hdb_df = calculate_amenity_metrics(hdb_df, bus_stops, 'bus_stop_dist', 'num_bus_stop_1km')

print("Processing Hawker Centres...")
hdb_df = calculate_amenity_metrics(hdb_df, hawkers, 'hawker_dist', 'num_hawker_1km')

print("Processing Parks...")
hdb_df = calculate_amenity_metrics(hdb_df, parks, 'park_dist', 'num_park_1km')

print("Processing Malls...")
hdb_df = calculate_amenity_metrics(hdb_df, malls, 'mall_dist', 'num_mall_1km')

# Save final result
hdb_df.to_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_bushawkerparkmall.csv', index=False)
print("Analysis complete! File saved as hdb_bushawkerparkmalls.csv")

Processing Bus Stops...
Processing Hawker Centres...
Processing Parks...
Processing Malls...
Analysis complete! File saved as hdb_bushawkerparkmalls.csv


In [54]:
prices = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/prices.csv')
flat_amenities = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_bushawkerparkmall.csv')
flatschools = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_school_proximity.csv')
flattopschools = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_top27school_proximity.csv')
flatmrt= pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_mrt_proximity.csv')
flatmrtinterchange= pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_mrtint_proximity.csv')


# To merge proximity data
df_proximity = flat_amenities.merge(flatschools, left_on='address', right_on='flat address', how='inner').drop(columns=['flat address'])
df_proximity = df_proximity.merge(flattopschools, left_on='address', right_on='flat address', how='inner').drop(columns=['flat address'])
df_proximity = df_proximity.merge(flatmrt, left_on='address', right_on='flat address', how='inner').drop(columns=['flat address'])
df_proximity = df_proximity.merge(flatmrtinterchange, left_on='address', right_on='flat address', how='inner').drop(columns=['flat address'])

# Final Merge
df_final = prices.merge(df_proximity, on='address', how='left')

# 3. Check the result
print(f"Final Dataframe Shape: {df_final.shape}")
display(df_final.head())

# 4. Export the final result back to your drive
df_final.to_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_final_combined.csv', index=False)

Final Dataframe Shape: (260568, 37)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,Nearest priority school,priority_school_dist_km,num_priority_sch_1km,num_priority_sch_2km,Nearest MRT Station,MRT_dist_km,num_mrt_1km,Nearest MRT interchange,MRT_interchange_dist_km,num_mrt_interchange_1km
0,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,07 TO 09,60.0,Improved,1986,70,...,CHIJ ST. NICHOLAS GIRLS' SCHOOL,0.415275,1.0,2.0,Mayflower,0.420669,1.0,Bishan,3.013362,0.0
1,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,01 TO 03,68.0,New Generation,1981,65,...,ROSYTH SCHOOL,2.132093,0.0,0.0,Ang Mo Kio,0.806901,1.0,Bishan,2.620788,0.0
2,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,01 TO 03,69.0,New Generation,1980,64,...,CHIJ ST. NICHOLAS GIRLS' SCHOOL,0.436188,1.0,2.0,Mayflower,0.292828,1.0,Bishan,2.831995,0.0
3,2015-01,ANG MO KIO,3 ROOM,446,ANG MO KIO AVE 10,01 TO 03,68.0,New Generation,1979,63,...,CATHOLIC HIGH SCHOOL,1.875405,0.0,1.0,Ang Mo Kio,0.688394,1.0,Bishan,1.952634,0.0
4,2015-01,ANG MO KIO,3 ROOM,557,ANG MO KIO AVE 10,07 TO 09,68.0,New Generation,1980,64,...,ROSYTH SCHOOL,1.898682,0.0,1.0,Ang Mo Kio,0.928377,1.0,Bishan,2.445208,0.0


In [55]:
#Add CPI for inflation as a feature

# Load and Process CPI
cpi_raw = pd.read_csv('/content/drive/MyDrive/GA Data analytics Capstone/Main CSV/ConsumerPriceIndexCPI2024AsBaseYearMonthly.csv')
all_items = cpi_raw[cpi_raw['DataSeries'].str.strip() == 'All Items']

# Get values for the last 3 months of 2025
val_oct = float(all_items['2025Oct'].iloc[0])
val_nov = float(all_items['2025Nov'].iloc[0])
val_dec = float(all_items['2025Dec'].iloc[0])

# Calculate average
avg_jan_2026 = round((val_oct + val_nov + val_dec) / 3, 3)
print(f"Calculated 2026-01 CPI: {avg_jan_2026:.3f}")

# Reshape CPI to "Long" format
years = range(2015, 2026)
months_short = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
target_cols = [f"{y}{m}" for y in years for m in months_short]
existing_cols = [c for c in target_cols if c in cpi_raw.columns]

df_cpi = all_items[['DataSeries'] + existing_cols].melt(id_vars=['DataSeries'], var_name='month_raw', value_name='cpi')


# Convert '2015Jan' to '2015-01'
month_map = {'Jan':'01', 'Feb':'02', 'Mar':'03', 'Apr':'04', 'May':'05', 'Jun':'06',
             'Jul':'07', 'Aug':'08', 'Sep':'09', 'Oct':'10', 'Nov':'11', 'Dec':'12'}
df_cpi['month'] = df_cpi['month_raw'].apply(lambda x: f"{x[:4]}-{month_map[x[4:]]}")
df_cpi = df_cpi[['month', 'cpi']]

# MERGE: Add CPI to HDB data
# We use 'left' join to keep all HDB transactions and just attach the CPI value for that month
df_model_ready = df_final.merge(df_cpi, on='month', how='left')

# Update the HDB dataframe
# This finds rows where month is '2026-01' and fills the 'cpi' column
df_model_ready.loc[df_model_ready['month'] == '2026-01', 'cpi'] = avg_jan_2026

# Check if adding 2026 Jan API worked
jan_2026_rows = df_model_ready[df_model_ready['month'] == '2026-01']
if not jan_2026_rows.empty:
    print(f"Successfully updated {len(jan_2026_rows)} rows for 2026-01.")
    print(jan_2026_rows[['month', 'cpi']].head())
else:
    print("Warning: No rows found for '2026-01'. Check your 'month' column format.")

# Verification
print(f"New Shape: {df_model_ready.shape}")
print("New Column Added: 'cpi'")
display(df_model_ready[['month', 'address', 'resale_price', 'cpi']].head())

# Save the final version
df_model_ready.to_csv('/content/drive/MyDrive/GA Data analytics Capstone/CSV from Jupyter/hdb_final_with_cpi.csv', index=False)

Calculated 2026-01 CPI: 101.588
Successfully updated 1332 rows for 2026-01.
          month      cpi
259236  2026-01  101.588
259237  2026-01  101.588
259238  2026-01  101.588
259239  2026-01  101.588
259240  2026-01  101.588
New Shape: (260568, 38)
New Column Added: 'cpi'


,month,address,resale_price,cpi
0,2015-01,174 ANG MO KIO AVE 4,255000.0,85.179
1,2015-01,541 ANG MO KIO AVE 10,275000.0,85.179
2,2015-01,163 ANG MO KIO AVE 4,285000.0,85.179
3,2015-01,446 ANG MO KIO AVE 10,290000.0,85.179
4,2015-01,557 ANG MO KIO AVE 10,290000.0,85.179
